In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
# Silver Layer — Clean, validate, normalize (matching Databricks logic exactly)
import re
from pyspark.sql.functions import col, when, trim, lower, initcap, expr, explode, split, dense_rank, regexp_replace, udf
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
from builtins import sum as py_sum


# 1. Load Bronze Tables
jobs        = spark.read.table("bronze.jobs")
locations   = spark.read.table("bronze.locations")
occupations = spark.read.table("bronze.occupations")
skills      = spark.read.table("bronze.skills")
channels    = spark.read.table("bronze.channels")
messages    = spark.read.table("bronze.messages")

# 2. Text Cleaning Helper Function
def clean_text(df, columns, use_initcap=False):
    for c in columns:
        cleaned = (
            when(~col(c).rlike("[a-zA-Z0-9]"), None)
            .when(trim(col(c)) == "", None)
            .when(lower(trim(col(c))).isin(
                "n/a", "not defined", "not specified",
                "undisclosed", "country name", "chet el", "null"), None)
        )
        if use_initcap:
            cleaned = cleaned.otherwise(initcap(trim(col(c))))
        else:
            cleaned = cleaned.otherwise(trim(col(c)))
        df = df.withColumn(c, cleaned)
    return df

jobs        = clean_text(jobs, ["job_name", "company_name", "job_descriptions"])
locations   = clean_text(locations, ["city", "country"], use_initcap=True)
locations   = clean_text(locations, ["full_address"])
occupations = clean_text(occupations, ["occupation"], use_initcap=True)
channels    = clean_text(channels, ["title", "description"])
messages    = clean_text(messages, ["content"])

# 3. Safe Date Casting
def safe_date(df, cols):
    for c in cols:
        if c in df.columns:
            df = df.withColumn(c, expr(f"try_cast({c} as date)"))
    return df

jobs     = safe_date(jobs, ["job_date", "created_at", "updated_at"])
channels = safe_date(channels, ["created_at", "updated_at"])
messages = safe_date(messages, ["created_at", "updated_at"])

# 4. Salary Cleaning UDF — EXACT copy from Databricks
def cleanSalary(salary_str):
    if salary_str is None:
        return None

    s = str(salary_str).lower().strip()

    if re.match(r'^[?\s\-\u2013\/\\|]+$', s):
        return None

    ignorable = ["kelishiladi", "negotiable", "belgilanmagan", "suhbat asosida",
                 "not specified", "to be discussed", "kelishilgan holda", "kelishamiz"]
    has_digits = any(ch.isdigit() for ch in s)
    if any(p in s for p in ignorable) and not has_digits:
        return None

    parts = re.split(r'[,;]\s*', s)
    valid_parts = [p for p in parts
                   if not any(kw in p for kw in ["donasiga", "sdelka", "shtuk", "dona"])]
    s_clean = " ".join(valid_parts) if valid_parts else s

    if not valid_parts:
        return None

    # FIX #1: Strip work-schedule text so those numbers don't leak into salary
    s_clean = re.sub(r'\d+\s*soatlik(\s*ish)?(\s*(vakti|kuni|grafik|smena))?', '', s_clean)
    s_clean = re.sub(r'soatlik\s+(ish\s*)?(vakti|kuni|grafik|smena)', '', s_clean)

    s_clean = re.sub(r'\d+/\d+', '', s_clean)
    s_clean = re.sub(r'\d+\s*%', '', s_clean)

    s_clean = re.sub(r'\s*[\.,]\s*', '.', s_clean)
    s_clean = re.sub(r'\.(?=\d{3}(?:\D|$))', '', s_clean)
    s_clean = re.sub(r'\s+(?=\d{3}(?:\D|$))', '', s_clean)

    raw_numbers = [float(n) for n in re.findall(r'\d+', s_clean)]
    if not raw_numbers:
        return None

    is_ming    = any(m in s for m in ["ming"])
    is_million = any(m in s for m in ["mln", "million", "млн"])
    is_usd     = any(u in s for u in ["$", "usd", "y.e"])
    is_rub     = any(r in s for r in ["rub", "rubl", "руб"])

    usd_rate = 12000.0
    rub_rate = 155.0

    has_large_som = any(v >= 10000 for v in raw_numbers) and not is_million and not is_ming

    valid_numbers = []
    for val in raw_numbers:
        if val < 1000:
            if is_ming:
                valid_numbers.append(val * 1000)
            elif (is_million or is_usd or is_rub) and not has_large_som:
                valid_numbers.append(val)
            else:
                continue
        else:
            valid_numbers.append(val)

    if not valid_numbers:
        return None

    converted = []
    for val in valid_numbers:
        if is_million:
            if val < 1000:
                val *= 1000000
            elif val < 10000:
                val *= 1000
        if is_usd and val < 15000:
            val *= usd_rate
        elif is_rub and val < 1000000:
            val *= rub_rate
        converted.append(val)

    if not converted:
        return None

    avg_salary = py_sum(converted) / len(converted)

    # FIX #2: Check hourly from s_clean (after schedule text removal), not raw s
    is_hourly = any(h in s_clean for h in ["hour", "soatbay", "soatiga", "per hour", "soatlik"])
    is_daily  = any(d in s for d in ["kunlik", "kuniga", "daily", "kunbay", "kun bay"])
    is_weekly = any(w in s for w in ["hafta", "haftalik", "week", "weekly"])
    is_yearly = any(y in s for y in ["yil", "yillik", "year", "yearly", "annually", "per year"])

    # FIX #3: Safety filters for large salaries
    if is_hourly and avg_salary > 500000:
        is_hourly = False
    if is_daily and avg_salary > 3000000:
        is_daily = False
    if is_weekly and avg_salary > 4000000:
        is_weekly = False
    if is_yearly and avg_salary < 30000000:
        is_yearly = False

    # FIX #4: Exclude USD/RUB from heuristic time-period conversion
    applied = False
    if is_hourly or (avg_salary < 15000 and not is_usd and not is_rub):
        avg_salary = avg_salary * 8 * 22
        applied = True
    elif is_daily or (30000 <= avg_salary < 800000 and not is_usd and not is_rub):
        avg_salary = avg_salary * 22
        applied = True
    elif is_weekly:
        avg_salary = avg_salary * 4.33
        applied = True
    elif is_yearly:
        avg_salary = avg_salary / 12.0
        applied = True

    # FIX #5: Final fallback also excludes USD/RUB
    if avg_salary < 800000 and not applied and not is_usd and not is_rub:
        avg_salary = avg_salary * 22

    # FIX #6: MAX_SALARY_LIMIT cap
    MAX_SALARY_LIMIT = 100_000_000.0
    if avg_salary > MAX_SALARY_LIMIT:
        return None

    return avg_salary

cleanSalary_udf = udf(cleanSalary, DoubleType())
jobs = jobs.withColumn("cleaned_salary", cleanSalary_udf(col("job_salary")))

# 5. Skills Normalization
df_normalized = (
    skills
    .withColumn("skill", explode(split(col("skill"), ",")))
    .withColumn("skill", lower(trim(col("skill"))))
    .withColumn("skill", regexp_replace(col("skill"), r'^[-"\'`]+|[-"\'`. ]+$', ''))
    .withColumn("skill", trim(col("skill")))
    .filter(
        (col("skill").isNotNull()) &
        (col("skill") != "") &
        (~col("skill").contains("?")) &
        (col("skill").rlike("[a-zA-Z]"))
    )
)
window_skill = Window.orderBy("skill")
norm_skills = df_normalized.withColumn("skill_id", dense_rank().over(window_skill))

# 6. Save to Silver Schema
jobs.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.jobs")
locations.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.locations")
occupations.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.occupations")
channels.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.channels")
messages.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.messages")
norm_skills.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.norm_skills")

print("--- Silver Layer Completed ---")

StatementMeta(, f57ad60e-d9c4-4348-a029-8865bdef8bb0, 3, Finished, Available, Finished, False)

--- Silver Layer Completed ---
